# L05 · Regularización y post-selección

**S5 · Lectura: Cap. 6 (+ Cap. 16) · 40–55 min**

## La pregunta

¿Puede una selección predictiva borrar un confusor?

## Qué harás

Comparar diferencia de medias, OLS y post-double-selection.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** ATE e intervalo con los tres métodos sobre Jobs.

> **Idea clave:** Para estimar un efecto, seleccionar variables solo por predicción puede ser insuficiente.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LinearRegression
rng = setup()
N_REP = R(60, 500)      # réplicas Monte Carlo (LAB_FULL=1 usa 500)
print("réplicas Monte Carlo:", N_REP)

## 1. El problema (§6.1)
Con muchas covariables ($p$ grande) no podemos ajustar por todas, así que **seleccionamos** con LASSO. El error típico es seleccionar mirando **solo a $Y$**: una covariable que predice mucho el tratamiento $D$ pero poco a $Y$ se descarta, y el efecto queda sesgado por esa **confusión omitida** aunque *parezca* irrelevante para $Y$.

Mundo simulado (Listado 6.1): $n=500$, $p$ covariables, 5 **confusores** que afectan a $D$ (fuerte) y a $Y$ (débil), efecto verdadero $\theta=1$.

In [ ]:
def generar(n, p, rng, theta=1.0, k=5, g_d=1.0, g_y=0.3):
    X = rng.normal(size=(n, p))
    D = X[:, :k] @ np.full(k, g_d) + rng.normal(size=n)
    Y = theta * D + X[:, :k] @ np.full(k, g_y) + rng.normal(size=n)
    return Y, D, X

def penalizacion(v, X, c=1.0):
    n, p = X.shape
    return c * np.std(v) * np.sqrt(2 * np.log(p) / n) / 2

def seleccionar(v, X, c=1.0):
    """Índices de columnas que LASSO deja en la regresión de v sobre X (X estandarizado)."""
    Z = (X - X.mean(0)) / X.std(0)
    m = Lasso(alpha=penalizacion(v, Z, c), max_iter=20000).fit(Z, v)
    return np.flatnonzero(m.coef_)

def ols_theta(y, d, X, S):
    A = np.column_stack([d, X[:, S]]) if len(S) else d[:, None]
    r = ols(y, A)
    return r["coef"][1], r["se"][1]

def seleccion_ingenua(y, d, X):
    """LASSO de Y sobre [D, X] con D sin penalizar: selecciona mirando SOLO a Y."""
    Z = (X - X.mean(0)) / X.std(0)
    m = Lasso(alpha=penalizacion(y, Z), max_iter=20000).fit(np.column_stack([d * 1e3, Z]), y)
    return np.flatnonzero(m.coef_[1:])

Y, D, X = generar(500, 100, rng)
S_ing = seleccion_ingenua(Y, D, X)
print("covariables que elige el método ingenuo:", S_ing.tolist() or "ninguna", "(los 5 confusores son las columnas 0–4)")
b, s = ols_theta(Y, D, X, S_ing)
print(f"θ̂ ingenuo = {fmt_ci(b, s)}   (θ verdadero = 1)")

**Mira el resultado.** El IC no contiene el valor verdadero. ¿Por qué el IC es *estrecho y equivocado* a la vez?

**Respuesta:**

Porque el sesgo por confusión omitida no se achica con más datos pero el error estándar sí: el IC se estrecha alrededor de un valor **equivocado**. La selección miró solo a $Y$ y descartó confusores que, dado $D$, parecían inútiles para $Y$.



## 2. Post-Double-Selection (§6.3 y Cap. 16 ◇)
**Idea:** seleccionar dos veces — covariables que predicen $Y$ **y** covariables que predicen $D$ — y ajustar por la **unión**. Un confusor pierde solo si es irrelevante para *ambos*, y entonces no sesga.
1. LASSO de $Y$ sobre $X$ → $S_y$.
2. LASSO de $D$ sobre $X$ → $S_d$.
3. OLS de $Y$ sobre $D$ y $X_{S_y\cup S_d}$.

In [ ]:
# ✏️ Completa tú — pds_theta
# Pista: np.union1d(Sy, Sd) y ols_theta(y, d, X, S).
def pds_theta(y, d, X):
    """Devuelve (theta, se, S) con S la UNIÓN de selecciones sobre Y y sobre D."""
    raise NotImplementedError("selecciona sobre y, selecciona sobre d, une, y ajusta OLS con ols_theta")

In [ ]:
b2, s2, S2 = pds_theta(Y, D, X)
print(f"PDS: {fmt_ci(b2, s2)}  con {len(S2)} covariables seleccionadas: {S2.tolist()}")
lab_check("pds_theta (selección)", lambda: set(pds_theta(Y, D, X)[2]) >= set(range(5)), [((), True, 0)])

**Partialling-out** (Listado 6.2) llega a lo mismo por otro camino: se quita de $Y$ y de $D$ lo que predicen las covariables y se regresa un residuo sobre otro (Frisch–Waugh–Lovell con selección).

In [ ]:
def partialling_out(y, d, X):
    Z = (X - X.mean(0)) / X.std(0)
    ry = y - Lasso(alpha=penalizacion(y, Z), max_iter=20000).fit(Z, y).predict(Z)
    rd = d - Lasso(alpha=penalizacion(d, Z), max_iter=20000).fit(Z, d).predict(Z)
    r = ols(ry, rd, const=False)
    return r["coef"][0], r["se"][0]
b3, s3 = partialling_out(Y, D, X)
print("partialling-out:", fmt_ci(b3, s3))

## 3. Monte Carlo: sesgo y cobertura al variar $p$ (ejercicio del capítulo)
Repetimos el experimento y medimos **sesgo** y **cobertura** (fracción de veces que el IC del 95 % contiene a $\theta=1$).

In [ ]:
def mc(p, n_rep, seed=0):
    r = np.random.default_rng(seed); filas = {"ingenuo": [], "PDS": [], "partialling-out": []}
    for _ in range(n_rep):
        Y, D, X = generar(500, p, r)
        bi, si = ols_theta(Y, D, X, seleccion_ingenua(Y, D, X)); filas["ingenuo"].append((bi, si))
        bp, sp, _ = pds_theta(Y, D, X); filas["PDS"].append((bp, sp))
        bo, so = partialling_out(Y, D, X); filas["partialling-out"].append((bo, so))
    out = []
    for k, v in filas.items():
        v = np.array(v); cob = np.mean(np.abs(v[:, 0] - 1) < 1.96 * v[:, 1])
        out.append({"p": p, "método": k, "sesgo": v[:, 0].mean() - 1, "cobertura": cob})
    return out
tab = pd.DataFrame([f for p in (20, 50, 100, 200) for f in mc(p, N_REP)])
show(tab.round(3))

In [ ]:
piv = tab.pivot(index="p", columns="método", values="cobertura")
piv.plot(marker="o"); plt.axhline(.95, ls="--", c="gray"); plt.ylim(0, 1.05); plt.title("Cobertura del IC al 95 % según p"); plt.show()

**Mira el resultado.** ¿Cambia el problema del método ingenuo al aumentar $p$? ¿Qué método mantiene la cobertura cerca del 95 %?

**Respuesta:**

El sesgo del ingenuo **no depende de $p$**: depende de que la selección ignore a $D$. PDS y partialling-out mantienen la cobertura cerca de 95 %. (Con pocas réplicas la cobertura es ruidosa: usa `LAB_FULL=1`.)



## 4. Penalizar sin cuidado (objetivo del sílabo)
Si metemos el tratamiento **dentro** del LASSO *como una covariable más*, la penalización **encoge su coeficiente** hacia 0: un sesgo introducido por el método, no por los datos.

In [ ]:
Y, D, X = generar(500, 100, np.random.default_rng(3), g_d=0.0)   # D aleatorizado: aquí no hay confusión
Z = (X - X.mean(0)) / X.std(0)
alphas = np.array([0.0005, 0.01, 0.05, 0.1, 0.3, 0.6])
coef_t = []
for a in alphas:
    m = Lasso(alpha=a, max_iter=20000).fit(np.column_stack([D / D.std(), Z]), Y)
    coef_t.append(m.coef_[0] / D.std())
plt.semilogx(alphas, coef_t, marker="o"); plt.axhline(1, c="gray", ls="--"); plt.xlabel("penalización α")
plt.ylabel("θ̂ (coeficiente de T)"); plt.title("Con T aleatorizado, el LASSO igual encoge θ̂"); plt.show()
print({f"α={a}": round(c, 3) for a, c in zip(alphas, coef_t)})

## 5. Aplicación al caso A1 (Jobs)
Comparamos diferencia en medias, OLS con todas las covariables y PDS. Recuerda: la diferencia entre ellos es **información sobre el método**, no sobre el supuesto de no confusión.

In [ ]:
sc = load_scenario("real/jobs"); sc.card()
X, y, d = sc.xyt("train")
est_dim, se_dim = diff_in_means(y, d)
r_ols = ols(y, np.column_stack([d, X])); est_ols, se_ols = r_ols["coef"][1], r_ols["se"][1]
est_pds, se_pds, S = pds_theta(y, d, X)
tabla = pd.DataFrame({"método": ["diferencia en medias", "OLS (todas las covariables)", "PDS"],
                      "ATE": [est_dim, est_ols, est_pds], "IC95% inf": [ci95(est_dim, se_dim)[0], ci95(est_ols, se_ols)[0], ci95(est_pds, se_pds)[0]],
                      "IC95% sup": [ci95(est_dim, se_dim)[1], ci95(est_ols, se_ols)[1], ci95(est_pds, se_pds)[1]]})
show(tabla.round(1))
print("covariables que PDS retiene:", [sc.covariates[j] for j in S])

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

In [ ]:
# EVALUATION-CELL  (solo aquí aparece la verdad)
print("ATE verdadero del escenario:", None if sc.true_ate is None else round(sc.true_ate, 1))
for k, e in zip(tabla["método"], tabla["ATE"]):
    print(f"  {k:30s} error = {e - sc.true_ate:+.1f}")

**Mira el resultado.** Mira los errores de la celda anterior. Si la diferencia en medias ya acierta (como ocurre en el Jobs del benchmark: 1600,8 frente a 1600,3), ¿qué aprendes?

**Respuesta:**

Que **en ese escenario** la asignación casi no depende de las covariables, y que PDS no tiene nada que corregir. Eso solo lo sabemos porque tenemos la verdad: con datos reales, la cercanía entre métodos no prueba el supuesto de no confusión, que sigue siendo **no observable**. Si, como en el stand-in, los métodos difieren, la diferencia habla del método, no de qué tan buena es la identificación.



## 6. Supuesto y bitácora

In [ ]:
fila = bitacora(
    pregunta="¿Cuál es el efecto del programa sobre el ingreso en Jobs y cambia según el método?",
    metodo="Post-Double-Selection (unión de selecciones sobre Y y sobre T)",
    estimando="ATE del programa sobre re78",
    numero_ic=fmt_ci(est_pds, se_pds, 1),
    supuesto=supuesto("no observable", "declarado y acotado", "no confusión condicional en las covariables medidas; los tres métodos coinciden pero eso no lo prueba"),
    decision="continuar", lab="L05")

---
## Qué queda para el libro
Ejercicios del libro cubiertos: variar p ∈ {20, 50, 100, 200}, implementar PDS y compararlo con partialling-out. Quedan: Ridge y el caso nhefs (el dataset no está en el benchmark). Las cifras del libro (sesgo −0,05…−0,10; cobertura 0,85–0,90) vienen de su propio DGP: aquí el DGP es otro y las cifras son las que ves.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*